# Exp 02: ResNet-18

目标：

1. 在更接近真实训练 workload 的 CNN 上观察 `torch.compile` 的收益。
2. 对比 eager mode 和不同 compile mode 的首次调用开销、稳态吞吐量。
3. 理解 BF16 AMP、CUDA Graphs、autotune、shape padding 对结果的影响。
4. 用 `counters` 观察固定 shape 下是否重复编译。

特殊环境变量：

```bash
# 看 graph break，ResNet-18 理论上应该比较少
TORCH_LOGS=graph_breaks python experiments_py/exp02_resnet.py

# 看编译器性能提示
TORCH_LOGS=perf_hints python experiments_py/exp02_resnet.py
```

硬件假设：NVIDIA GPU with CUDA，PyTorch 2.x，已安装 `torchvision`。


## 准备工作

这个实验和 Toy MLP 的区别是：ResNet-18 里有卷积、BatchNorm、ReLU、残差连接和 pooling，计算图更接近真实 CV 模型。

注意：notebook 中的 benchmark 容易受 GPU clock、温度、后台进程、编译缓存影响。这里会做多轮测量并取 median，但它仍然是一个教学实验，不是严格的 production benchmark。


In [1]:
import gc
import statistics
import time

import torch
import torch.nn.functional as F
import torchvision.models as tvm
from torch._dynamo.utils import counters

In [2]:
assert torch.cuda.is_available(), "这个实验需要 CUDA GPU"

print(f"torch      : {torch.__version__}")
print(f"torchvision: {tvm.__name__}")
print(f"device     : {torch.cuda.get_device_name(0)}")

torch      : 2.10.0+cu128
torchvision: torchvision.models
device     : NVIDIA L4


## 超参数

默认 batch size 设为 64，输入是 ImageNet 标准的 `3x224x224`。如果显存不够，可以先把 `BATCH` 改成 16 或 32。

`REPEATS` 用来缓解 notebook benchmark 的随机性：每个配置重复测几次，最后取 median。


In [3]:
DEVICE = "cuda"
DTYPE = torch.bfloat16
BATCH = 64
IMG_SIZE = 224
NUM_CLS = 1000
WARMUP = 3
MEASURE = 20
REPEATS = 3

# max-autotune 对 ResNet 编译可能明显更久；需要时再打开。
INCLUDE_MAX_AUTOTUNE = False

torch.manual_seed(42)
torch.set_float32_matmul_precision("high")

## 工具函数

这里统计的是训练 step 的吞吐量：每一步包含 forward、cross entropy loss、backward、清梯度，但不包含 optimizer step。

为什么不用 CPU 计时器直接包住循环？因为 CUDA 操作默认异步，CPU 提交完 kernel 可能马上继续执行。这里仍然用 CUDA Event 统计 GPU 时间轴上的耗时，再换算成 `samples/sec`。


In [4]:
def make_model() -> torch.nn.Module:
    return tvm.resnet18(weights=None).to(DEVICE)


def make_batch(batch_size: int = BATCH):
    x = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    y = torch.randint(0, NUM_CLS, (batch_size,), device=DEVICE)
    return x, y


def clear_grads(model: torch.nn.Module) -> None:
    for p in model.parameters():
        p.grad = None


def train_step(
    model: torch.nn.Module, x: torch.Tensor, y: torch.Tensor
) -> torch.Tensor:
    with torch.autocast("cuda", dtype=DTYPE):
        loss = F.cross_entropy(model(x), y)
    loss.backward()
    clear_grads(model)
    return loss


def time_throughput(model: torch.nn.Module, n_steps: int = MEASURE) -> float:
    """Return training throughput in samples/sec for fwd+bwd+zero_grad."""
    x, y = make_batch()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    torch.cuda.synchronize()
    start.record()
    for _ in range(n_steps):
        train_step(model, x, y)
    end.record()
    torch.cuda.synchronize()

    total_ms = start.elapsed_time(end)
    return (BATCH * n_steps) / (total_ms / 1000)


def warmup(model: torch.nn.Module, n_steps: int = WARMUP) -> None:
    for _ in range(n_steps):
        x, y = make_batch()
        train_step(model, x, y)
    torch.cuda.synchronize()


def median_throughput(
    model: torch.nn.Module, repeats: int = REPEATS
) -> tuple[float, list[float]]:
    values = [time_throughput(model, MEASURE) for _ in range(repeats)]
    return statistics.median(values), values


def cleanup_cuda() -> None:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

## Eager Mode Baseline

先跑普通 PyTorch eager mode，作为后续 compile 的 baseline。

这里使用 BF16 autocast，因为现代 NVIDIA GPU 上 BF16 通常能走 Tensor Core，训练吞吐量会更接近真实设置。


In [5]:
base_model = make_model()
ref_state = {k: v.detach().clone() for k, v in base_model.state_dict().items()}

warmup(base_model)
eager_tput, eager_runs = median_throughput(base_model)

print(f"Eager runs : {[round(v, 1) for v in eager_runs]} samples/sec")
print(f"Eager median: {eager_tput:.1f} samples/sec")

Eager runs : [1390.3, 1388.2, 1387.1] samples/sec
Eager median: 1388.2 samples/sec


## First Compile Overhead

第一次调用 compiled model 时，通常不只是运行一次 forward/backward，还会触发 Dynamo trace、AOTAutograd、Inductor codegen、kernel 编译和缓存写入。

所以这里用 `time.perf_counter()` 测的是首个 call 的端到端墙钟时间：它主要包含 CPU 侧编译工作，也包含最后 `torch.cuda.synchronize()` 等待 GPU 首次执行完成的时间。


In [6]:
model_c = make_model()
model_c.load_state_dict(ref_state)
counters.clear()

compiled_default = torch.compile(model_c, mode="default", fullgraph=False)

x, y = make_batch()
t0 = time.perf_counter()
train_step(compiled_default, x, y)
torch.cuda.synchronize()
first_call_ms = (time.perf_counter() - t0) * 1000

print(f"首次 call（含编译）: {first_call_ms:.1f} ms")
print(f"unique_graphs after 1st call: {counters['stats']['unique_graphs']}")

首次 call（含编译）: 3456.2 ms
unique_graphs after 1st call: 1


为什么你明明用了 mode="default" 也会看到上面有一个 max_autotune_gemm 的 warning 呢？

因为 torch.compile(..., mode="default") 也会经过 Inductor 的优化决策。某些内部配置、默认 heuristic、环境状态或 PyTorch 版本行为可能会检查 max-autotune GEMM 是否可用，即使最后没有真的走完整 max-autotune。

所以它更像是 Inductor 的“性能提示日志”，不是说你显式开了 mode="max-autotune"。


## Compile Mode 对比

这一步比较不同 `mode` 的稳态吞吐量。需要注意：notebook 反复运行时，Inductor 可能复用已经生成的缓存，所以这里的“首次 call”更适合理解为当前会话下的 first call wall time，不一定等价于完全 cold compile。


In [7]:
modes = [
    "default",
    "reduce-overhead",
    "max-autotune-no-cudagraphs",
]

if INCLUDE_MAX_AUTOTUNE:
    modes.append("max-autotune")

results = {}

for mode in modes:
    cleanup_cuda()
    m = make_model()
    m.load_state_dict(ref_state)
    counters.clear()

    compiled = torch.compile(m, mode=mode, fullgraph=False)

    x, y = make_batch()
    t0 = time.perf_counter()
    train_step(compiled, x, y)
    torch.cuda.synchronize()
    first_ms = (time.perf_counter() - t0) * 1000

    warmup(compiled)
    median_tput, runs = median_throughput(compiled)
    graphs = counters["stats"]["unique_graphs"]

    results[mode] = {
        "first_ms": first_ms,
        "median_tput": median_tput,
        "runs": runs,
        "graphs": graphs,
    }

    print(
        f"[{mode:30s}] first={first_ms:8.1f} ms  "
        f"median={median_tput:8.1f} samples/sec  "
        f"speedup={median_tput / eager_tput:5.2f}x  "
        f"graphs={graphs}  runs={[round(v, 1) for v in runs]}"
    )

[default                       ] first=    32.3 ms  median=  1907.1 samples/sec  speedup= 1.37x  graphs=0  runs=[1935.1, 1907.1, 1902.6]


W0425 09:55:16.347000 59333 torch/_inductor/utils.py:1679] [0/1] Not enough SMs to use max_autotune_gemm mode


[reduce-overhead               ] first= 17984.5 ms  median=  1891.7 samples/sec  speedup= 1.36x  graphs=1  runs=[1922.7, 1884.0, 1891.7]


/usr/local/lib/python3.12/dist-packages/torch/_inductor/select_algorithm.py:3464: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  current_size = base.storage().size()
Autotune Choices Stats:
{"num_choices": 2, "num_triton_choices": 0, "best_kernel": "bias_addmm", "best_time": 0.01532800029963255}
AUTOTUNE addmm(64x1000, 64x512, 512x1000)
strides: [0, 1], [512, 1], [1, 512]
dtypes: torch.bfloat16, torch.bfloat16, torch.bfloat16
  bias_addmm 0.0153 ms 100.0% 
  addmm 0.0184 ms 83.2% 
SingleProcess AUTOTUNE benchmarking takes 0.0896 seconds and 0.0003 seconds precompiling for 2 choices


[max-autotune-no-cudagraphs    ] first=162133.8 ms  median=  1935.7 samples/sec  speedup= 1.39x  graphs=1  runs=[1966.3, 1924.5, 1935.7]


## `shape_padding` 实验

`shape_padding=True` 会让 Inductor 在合适的地方把维度 pad 到更利于 Tensor Core / vectorized kernel 的形状。它不是保证一定更快的开关，但在 BF16、固定 shape、某些 convolution/GEMM 组合上可能有帮助。

这里把它作为一个可选实验：如果当前 PyTorch 版本不支持某个 option，或者收益不明显，都属于正常现象。


In [10]:
cleanup_cuda()

try:
    m_pad = make_model()
    m_pad.load_state_dict(ref_state)
    counters.clear()

    # 当前 PyTorch 版本不允许同时传 mode 和 options。
    # 不传 mode 时使用默认 compile 行为，再叠加这里显式给出的 Inductor options。
    compiled_pad = torch.compile(
        m_pad,
        fullgraph=False,
        options={"shape_padding": True, "epilogue_fusion": True},
    )

    x, y = make_batch()
    t0 = time.perf_counter()
    train_step(compiled_pad, x, y)
    torch.cuda.synchronize()
    first_pad_ms = (time.perf_counter() - t0) * 1000

    warmup(compiled_pad)
    pad_tput, pad_runs = median_throughput(compiled_pad)

    default_tput = results["default"]["median_tput"]
    print(f"default                 : {default_tput:.1f} samples/sec")
    print(
        f"default + shape_padding : {pad_tput:.1f} samples/sec ({pad_tput / default_tput:.2f}x vs default)"
    )
    print(f"first call              : {first_pad_ms:.1f} ms")
    print(f"runs                    : {[round(v, 1) for v in pad_runs]}")
    print(f"unique_graphs           : {counters['stats']['unique_graphs']}")
except Exception as exc:
    print("shape_padding 实验跳过：", type(exc).__name__, exc)

default                 : 1907.1 samples/sec
default + shape_padding : 1884.2 samples/sec (0.99x vs default)
first call              : 1435.5 ms
runs                    : [1906.2, 1884.2, 1884.0]
unique_graphs           : 1


## 结果汇总

这个表只汇总当前 notebook 会话中的结果。若你多次运行，数字可能变化；更应该关注趋势：哪些 mode 稳定快、哪些 mode 编译时间明显长、`unique_graphs` 是否保持在 1。


In [11]:
print(f"eager baseline: {eager_tput:.1f} samples/sec")
print("-" * 88)
print(
    f"{'mode':32s} {'median smp/s':>14s} {'speedup':>9s} {'first ms':>12s} {'graphs':>8s}"
)
print("-" * 88)

for mode, item in results.items():
    print(
        f"{mode:32s} "
        f"{item['median_tput']:14.1f} "
        f"{item['median_tput'] / eager_tput:8.2f}x "
        f"{item['first_ms']:12.1f} "
        f"{item['graphs']:8d}"
    )

eager baseline: 1388.2 samples/sec
----------------------------------------------------------------------------------------
mode                               median smp/s   speedup     first ms   graphs
----------------------------------------------------------------------------------------
default                                  1907.1     1.37x         32.3        0
reduce-overhead                          1891.7     1.36x      17984.5        1
max-autotune-no-cudagraphs               1935.7     1.39x     162133.8        1


## 小结

这个 ResNet-18 实验比 Toy MLP 更接近真实模型，但结论仍然需要按 workload 解读：

- 首次 compiled call 往往远慢于单个 eager step，因为它包含编译和首次执行。
- 固定 batch、固定 image size 时，`unique_graphs` 理想情况下应接近 1；如果它持续增加，说明发生了 recompile。
- ResNet-18 主要由 conv / batchnorm / relu / residual add 组成，`torch.compile` 可能通过 fusion、layout、autotune、减少 launch overhead 获得稳态收益。
- `default` 是最稳妥的起点；`reduce-overhead` 适合固定 shape 且 CPU launch overhead 明显的场景；`max-autotune`/`max-autotune-no-cudagraphs` 适合愿意用更长编译时间换稳态性能的场景。
- 如果不同轮次抖动较大，应增加 `MEASURE` 和 `REPEATS`，观察 median，而不是只看单次结果。

下一步可以进入 dynamic shape 实验：让 batch size 或 image size 改变，观察 recompile 和 guard 如何影响性能。
